# Week 6 Hand-in: Open Economy, Exchange Rates and the Peg

**Macroeconomics B -- Lectures 9 and 10**

This notebook is the starting point for the programming part of the hand-in. Write short Markdown answers after the code cells. Keep the answers concise and refer to your own numbers.

The notebook is self-contained. The small inflation panel below is a stylised Denmark/euro-area example used to practise the mechanism under a fixed nominal exchange rate.

## 1. Setup

We use the lecture notation:

$ i_t = i_t^f + \Delta e^e_{t+1}$,  
$ \Delta e^r_t = \Delta e_t + \pi^f_t - \pi_t$,  
$ \widehat y_t = \beta_1(q_{t-1}-\widehat\pi_t)+z_t$,  
$ \widehat\pi_t = \gamma \widehat y_t+s_t$,  
$ q_t = q_{t-1}-\widehat\pi_t$ under a fixed nominal exchange rate.

Here $q_t$ is the real-exchange-rate gap and $\widehat\pi_t=\pi_t-\pi^f$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linestyle': '--',
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
})

par = {
    'i_f': 0.025,   # foreign nominal interest rate, decimal
    'beta1': 0.70,  # trade/competitiveness channel in the fixed-peg AD curve
    'gamma': 0.50,  # AS slope
    'a_closed': 1.20,
    'beta2': 0.40,  # used ONLY in Part 1 Q4 (analytical); in Part 3-5 simulations beta2 is absorbed into z_t (cf. Lecture 10 Appendix A)
}

## 2. UIP, credibility and expected devaluation

Complete the functions and produce a table for the four scenarios from Part 1 of the hand-in.

Use decimals in the code. For example, 2.5 percent is `0.025`.

In [ ]:
def uip_rate_log(i_f, expected_depreciation):
    # Log-linear UIP: i = i_f + expected depreciation.
    # TODO: replace np.nan by the correct expression.
    return np.nan


def uip_rate_exact(i_f, expected_log_depreciation):
    # Exact UIP when expected depreciation is measured as a log change:
    # 1+i = (1+i_f)*exp(expected_log_depreciation).
    # TODO: replace np.nan by the correct expression.
    return np.nan

scenarios = pd.DataFrame({
    'scenario': ['credible peg', 'mild pressure', 'severe pressure', 'expected appreciation'],
    'expected_depreciation': [0.00, 0.20 * 0.06, 0.40 * 0.10, -0.01],
})

# TODO: add log-linear and exact domestic rates to the table.
scenarios

In [ ]:
# Plot the UIP schedule and compare the log-linear and exact version.
# TODO: create a grid for expected depreciation from -2% to 6%, compute both rates, and plot them.

**Interpretation.** In two or three sentences, explain why expected devaluation raises the domestic rate under a peg, and why this can be contractionary before any devaluation occurs.

## 3. Real exchange rate and Marshall-Lerner

A fixed nominal exchange rate means $\Delta e_t=0$, so the real exchange rate moves with the inflation differential.

In [ ]:
def real_exchange_rate_change(nominal_depreciation, pi_foreign, pi_domestic):
    # Return Delta q = Delta e + pi_f - pi, using percentage points.
    # TODO
    return np.nan


def ml_trade_balance_effect(real_depreciation, eps_x, eps_m, quantity_adjustment=1.0):
    # Approximate change in NX divided by initial imports, in percent.
    # Starting from balanced trade:
    # dNX/(E*M) = (quantity_adjustment*(eps_x+eps_m) - 1) * dE^r/E^r.
    # real_depreciation is measured in percent, e.g. 5 for 5 percent.
    # TODO
    return np.nan

ml_cases = pd.DataFrame({
    'eps_x': [0.4, 0.7, 1.2],
    'eps_m': [0.3, 0.5, 0.6],
})
# TODO: compute the long-run trade-balance effect of a 5% real depreciation.
ml_cases

In [ ]:
# J-curve exercise: compare the first-quarter and long-run effects.
# Use eps_x + eps_m = 1.2, a 5% real depreciation, and first-quarter adjustment = 0.2.

## 4. A stylised Denmark/euro-area inflation differential

The data below are embedded to keep the notebook portable. Treat them as a stylised monthly path for year-on-year inflation rates.

**Note on notation.** In this section we build a *real-exchange-rate level index* with Jan 2021 = 100. This is NOT the same object as the model gap $q_t$ used in Section 5 (which is a deviation from long-run equilibrium). The driving formula is the same identity $\Delta e^r_t = \pi^f_t-\pi_t$, just expressed in monthly index points:

$\text{rer\_index}_t = \text{rer\_index}_{t-1} + (\pi^f_t - \pi_t)/12.$

We divide by 12 because the inflation series is annual % year-on-year and we want a monthly increment.

In [ ]:
def make_inflation_panel():
    dates = pd.date_range('2021-01-01', '2026-04-01', freq='MS')
    t = np.arange(len(dates))
    dk = np.interp(t, [0, 12, 24, 36, 48, 60, 63], [0.4, 1.8, 8.5, 3.3, 1.3, 1.6, 1.2])
    ea = np.interp(t, [0, 12, 24, 36, 48, 60, 63], [0.3, 2.6, 8.4, 5.4, 2.4, 2.2, 3.0])
    return pd.DataFrame({'Denmark': dk, 'Euro area': ea}, index=dates)

infl = make_inflation_panel()
infl.tail()

In [ ]:
# TODO: construct a real-exchange-rate LEVEL index (call it e.g. rer_index) with Jan 2021 = 100.
# Use monthly_change = (Euro area inflation - Denmark inflation)/12 and cumulate.
# Then plot Denmark inflation, euro-area inflation, and the rer_index in separate figures.
# Reminder: rer_index here is a level series, not the model gap q_t from Section 5.

**Interpretation.** Does the stylised path imply an improvement or deterioration in Danish price competitiveness after 2023? Use the sign of $\pi^f-\pi$ in your answer.

## 5. Fixed-rate AS-AD simulation

Complete the simulation function. The timing is:

1. inherit $q_{t-1}$,
2. solve AD and AS for $\widehat y_t$ and $\widehat\pi_t$,
3. update $q_t=q_{t-1}-\widehat\pi_t$.

In [ ]:
def simulate_peg(T, beta1, gamma, z_path=None, s_path=None, q0=0.0):
    if z_path is None:
        z_path = np.zeros(T)
    if s_path is None:
        s_path = np.zeros(T)
    z_path = np.asarray(z_path, dtype=float)
    s_path = np.asarray(s_path, dtype=float)

    yhat = np.empty(T)
    pihat = np.empty(T)
    q = np.empty(T)
    q_lag = q0

    for t in range(T):
        # TODO: implement the within-period solution.
        # yhat[t] = ...
        # pihat[t] = ...
        # q[t] = ...
        yhat[t] = np.nan
        pihat[t] = np.nan
        q[t] = np.nan
        q_lag = q[t]

    return pd.DataFrame({
        'period': np.arange(1, T + 1),
        'z': z_path,
        's': s_path,
        'yhat': yhat,
        'pihat': pihat,
        'q': q,
    })

# Temporary fiscal expansion: z_1 = 1, z_t = 0 afterwards.
T = 20
z_path = np.zeros(T)
z_path[0] = 1.0
peg = simulate_peg(T, par['beta1'], par['gamma'], z_path=z_path)
peg.head()

In [ ]:
# TODO: plot yhat, pihat and q from the fixed-peg simulation.
# TODO: identify the first period after period 1 in which yhat is negative.

In [ ]:
# TODO: Repeat the simulation for beta1 = 0.4, 0.7, 1.0.
# Compute the convergence root 1/(1+beta1*gamma) and the half-life log(0.5)/log(root).

## 6. Comparison with the closed-economy AS-AD model

In the closed-economy model with static expectations:

$ \widehat y_t = z_t - a\widehat\pi_t$,  
$ \widehat\pi_t = \widehat\pi_{t-1} + \gamma\widehat y_t$.

The same temporary demand expansion can create a later downturn, but the state variable is different.

**Note on $\widehat\pi_t$.** In the fixed-peg model of Sections 3 and 5, $\widehat\pi_t = \pi_t - \pi^f$ (deviation from foreign inflation, the peg's nominal anchor). In the closed-economy model here there is no foreign anchor, so $\widehat\pi_t$ is the deviation from the steady-state inflation target. Static expectations means $\pi^e_t = \pi_{t-1}$, which produces the lagged term in AS. The slope $a$ is the reduced-form sensitivity of demand to inflation (it bundles the real-rate channel and a Taylor-type monetary response).

In [ ]:
def simulate_closed(T, a, gamma, z_path=None, s_path=None, pihat0=0.0):
    if z_path is None:
        z_path = np.zeros(T)
    if s_path is None:
        s_path = np.zeros(T)
    z_path = np.asarray(z_path, dtype=float)
    s_path = np.asarray(s_path, dtype=float)

    yhat = np.empty(T)
    pihat = np.empty(T)
    pi_lag = pihat0

    for t in range(T):
        # TODO: solve the two equations for yhat[t] and pihat[t].
        yhat[t] = np.nan
        pihat[t] = np.nan
        pi_lag = pihat[t]

    return pd.DataFrame({'period': np.arange(1, T + 1), 'z': z_path, 's': s_path,
                         'yhat': yhat, 'pihat': pihat})

closed = simulate_closed(T, par['a_closed'], par['gamma'], z_path=z_path)
closed.head()

In [ ]:
# TODO: plot the output-gap paths from the fixed-peg and closed-economy simulations in one figure.

**Interpretation.** Explain the difference between the two propagation mechanisms. In the fixed-peg model the inherited state is $q_{t-1}$; in the closed-economy model it is lagged expected inflation.

## 7. Short policy memo

Write 250--350 words. Use the numerical results above and answer the memo question from Part 5 of the hand-in.